# Working reference · QAOA for two regimes and two policies

This experiment assigns a policy (nominal or robust) to each of two operating regimes. The default matrix is illustrative. To use measured IE costs, run the offline evaluator from the repository root and set `instance_path` below to the JSON path relative to the repository, with no leading slash.

```text
python ie/run_synthesis_offline.py --seeds 42
```

Omit `--seeds 42` for five matched seeds. The offline instance uses zero switching cost. Its matrix stores the mean cost for each regime and policy.

Install the project with `python -m pip install -r requirements.txt` from the repo root and select that environment's Python kernel. The circuit runs locally on Qiskit Aer.

In [2]:
from itertools import product
import json
from pathlib import Path

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator

instance_path = None  # Use the built-in example.
instance_path = Path('ie/results/runs/offline_20261122T223905032910Z/instance.json')
if instance_path is None:
    costs = np.array([[1.0, 3.0], [2.5, 1.0]])
    switch_cost = 2.0
    penalty = 5.0
    policies = ("nominal", "robust")
    angle_scale = 1.0
else:
    repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                      if (path / 'ie' / 'run_synthesis_offline.py').is_file()), None)
    if repo_root is None:
        raise FileNotFoundError('Start the notebook kernel inside the Q-SYNTH checkout.')
    instance = json.loads((repo_root / instance_path).read_text(encoding="utf-8"))
    if instance.get("schema_version") != 1:
        raise ValueError("Expected a version 1 IE instance.")
    costs = np.array(instance["cost_matrix"], dtype=float)
    switch_cost = float(instance["switch_cost"])
    policies = tuple(policy["name"] for policy in instance["policies"])
    if costs.shape != (2, 2) or not np.isfinite(costs).all() or not np.isfinite(switch_cost):
        raise ValueError("Expected four finite costs and a finite switching cost.")
    angle_scale = max(1.0, np.max(np.abs(costs)), abs(switch_cost))
    # Bound the full range of the unpenalized binary objective.
    penalty = np.abs(costs).sum() + 2 * abs(switch_cost) + angle_scale
    print("Regimes in row order:", [regime["name"] for regime in instance["regimes"]])
    print("Cost unit:", instance["cost_unit"])

assignments = {
    (a, b): costs[0, a] + costs[1, b] + switch_cost * (a != b)
    for a, b in product(range(2), repeat=2)
}
best_cost = min(assignments.values())
optimal = {pair for pair, value in assignments.items() if np.isclose(value, best_cost)}
for pair, value in sorted(assignments.items(), key=lambda item: item[1]):
    print(tuple(policies[i] for i in pair), value)


Regimes in row order: ['baseline_noise', 'elevated_noise']
Cost unit: kBT/s
('nominal', 'robust') -137.95940960779922
('robust', 'robust') -136.40195674204995
('nominal', 'nominal') -135.90467115182773
('robust', 'nominal') -134.34721828607846


## Check the QUBO and Ising encoding

Use one bit for each regime–policy pair, ordered by qubit index as (regime 0 nominal, regime 0 robust, regime 1 nominal, regime 1 robust). Each regime gets a quadratic penalty of $A(q_0+q_1-1)^2$. The switching cost multiplies the two cross-regime terms that select different policies. Substitute $q_i=(1-Z_i)/2$ and check every computational basis state.

In [3]:
# Use one coefficient per pair i < j: E(q) = c + sum(l_i q_i) + sum(w_ij q_i q_j).
linear = costs.ravel() - penalty
quadratic = {
    (0, 1): 2 * penalty,
    (2, 3): 2 * penalty,
    (0, 3): switch_cost,
    (1, 2): switch_cost,
}
offset = 2 * penalty + linear.sum() / 2
fields = -linear / 2
couplings = {}
for (i, j), weight in quadratic.items():
    offset += weight / 4
    fields[i] -= weight / 4
    fields[j] -= weight / 4
    couplings[i, j] = weight / 4

hamiltonian = SparsePauliOp.from_sparse_list(
    [("Z", [i], h) for i, h in enumerate(fields)]
    + [("ZZ", [i, j], value) for (i, j), value in couplings.items()],
    num_qubits=4,
)
energies = hamiltonian.to_matrix().diagonal().real + offset

def decode(bits):
    if sum(bits[:2]) == 1 and sum(bits[2:]) == 1:
        return bits[1], bits[3]
    return None

# Qiskit labels basis states with the highest-index qubit on the left.
labels = [f"{index:04b}" for index in range(16)]
choices = []
for label, energy in zip(labels, energies):
    bits = tuple(map(int, reversed(label)))
    direct = (
        costs.ravel() @ bits
        + switch_cost * (bits[0] * bits[3] + bits[1] * bits[2])
        + penalty * ((sum(bits[:2]) - 1)**2 + (sum(bits[2:]) - 1)**2)
    )
    assert np.isclose(energy, direct), label
    choice = decode(bits)
    if choice is not None:
        assert np.isclose(energy, assignments[choice]), label
    choices.append(choice)

ground_choices = {
    choice for choice, energy in zip(choices, energies)
    if np.isclose(energy, energies.min())
}
assert ground_choices == optimal
print("All 16 energies verified. Ground cost:", energies.min())


All 16 energies verified. Ground cost: -137.95940960779922


## Simulate depth-one QAOA

Scan a finite grid of angles using ideal statevector expectations. Then run the selected circuit with 4,096 shots on Aer. The constant Ising offset contributes only a global phase to the circuit and is included when reporting energies.

In [4]:
def qaoa_circuit(gamma, beta):
    circuit = QuantumCircuit(4)
    circuit.h(range(4))
    for i, h in enumerate(fields):
        circuit.rz(2 * gamma * h, i)
    for (i, j), value in couplings.items():
        circuit.rzz(2 * gamma * value, i, j)
    circuit.rx(2 * beta, range(4))
    return circuit

def expectation(gamma, beta):
    state = Statevector.from_instruction(qaoa_circuit(gamma, beta))
    return float(state.probabilities() @ energies)

grid = product(
    np.linspace(0, 2 * np.pi / angle_scale, 48, endpoint=False),
    np.linspace(0, np.pi, 32, endpoint=False),
)
expected_energy, gamma, beta = min(
    (expectation(gamma, beta), gamma, beta) for gamma, beta in grid
)
circuit = qaoa_circuit(gamma, beta)
measured = circuit.copy()
measured.measure_all()
backend = AerSimulator()
compiled = transpile(measured, backend, seed_transpiler=42)
shots = 4096
counts = backend.run(compiled, shots=shots, seed_simulator=42).result().get_counts()
assert sum(counts.values()) == shots

feasible = np.array([choice is not None for choice in choices])
is_optimal = np.array([choice in optimal for choice in choices])
ideal = Statevector.from_instruction(circuit).probabilities()
sampled = np.array([counts.get(label, 0) / shots for label in labels])
sampled_costs = [assignments[choice] for choice, mass in zip(choices, sampled)
                 if choice is not None and mass > 0]

print(f"Grid-selected angles: gamma={gamma:.4f}, beta={beta:.4f}")
print(f"Expected penalized cost: {expected_energy:.3f} (ideal), {sampled @ energies:.3f} (shots)")
print(f"Feasible probability: {ideal @ feasible:.1%} (ideal), {sampled @ feasible:.1%} (shots)")
print(f"Optimal probability: {ideal @ is_optimal:.1%} (ideal), {sampled @ is_optimal:.1%} (shots)")
print("Best sampled feasible cost:", min(sampled_costs, default=None))
print("Exact feasible cost:", best_cost)
print("Circuit depth before measurement:", circuit.depth())


Grid-selected angles: gamma=0.0036, beta=2.7489
Expected penalized cost: -105.808 (ideal), -106.854 (shots)
Feasible probability: 89.3% (ideal), 89.6% (shots)
Optimal probability: 22.4% (ideal), 23.0% (shots)
Best sampled feasible cost: -137.95940960779922
Exact feasible cost: -137.95940960779922
Circuit depth before measurement: 5


The grid search minimizes expected penalized energy, so it can leave probability on infeasible strings. Compare the sampled optimum with the exact enumeration and keep the feasibility rate in view when replacing the example matrix with measured IE costs.